# Notebook 3 — API FastAPI que Lee de MongoDB
**Proyecto:** Ernesto Investing AI — iDeSo

Esta API **no recalcula nada pesado** (ni SVC, ni RNN, ni LSTM): solo lee de las colecciones ya pobladas por los Notebooks 1, 2, 4 y 5. La única excepción es `/api/portafolio/optimizar`, que sí calcula al vuelo porque es una optimización numérica barata (no un entrenamiento de modelo).

Endpoints:
- `GET /api/salud`
- `GET /api/mercado/{ticker}`
- `GET /api/svc/{ticker}`
- `GET /api/rnns/{ticker}` — clasificadores RNN (Notebook 4)
- `GET /api/lstm/{ticker}` — regresor LSTM, horizonte fijo de 30 días (Notebook 5)
- `POST /api/auth/registro`, `POST /api/auth/login`
- `POST /api/portafolio/optimizar` — Markowitz, calculado al vuelo con `scipy`

Se expone a Internet con **ngrok**, para que el frontend (GitHub Pages) pueda hacerle `fetch()`.

## 1. Instalación de dependencias

In [28]:
!pip install -q fastapi uvicorn pyngrok pymongo[srv] nest-asyncio scipy
!pip install certifi


In [29]:
from google.colab import userdata

MONGO_URI = userdata.get('MONGO_URI')
NGROK_AUTHTOKEN = userdata.get('NGROK_AUTHTOKEN')

if MONGO_URI and NGROK_AUTHTOKEN:
    print('✅ Variables cargadas correctamente y listas para usarse')
else:
    print('❌ Faltan datos: agrega MONGO_URI y NGROK_AUTHTOKEN en el panel de Secrets de Colab (icono de llave)')


✅ Variables cargadas correctamente y listas para usarse


## 3. Conexión a MongoDB Atlas

In [30]:
import certifi
from pymongo import MongoClient

# Le pasamos certifi para evitar errores de certificados SSL
cliente = MongoClient(MONGO_URI, tlsCAFile=certifi.where())

db = cliente['ernesto_investing_ai']

col_precios = db['precios_ohlcv']
col_predicciones = db['predicciones']
col_metricas = db['metricas_modelos']

# Verificacion rapida de conexion
cliente.admin.command('ping')
print('✅ Conexion a MongoDB Atlas exitosa')
print('Documentos en precios_ohlcv:', col_precios.count_documents({}))
print('Documentos en predicciones:', col_predicciones.count_documents({}))
print('Documentos en metricas_modelos:', col_metricas.count_documents({}))
col_rnns = db['clasificaciones_rnn']    # poblada por el Notebook 4
col_lstm = db['predicciones_lstm']      # poblada por el Notebook 5
col_usuarios = db['usuarios']           # poblada por esta misma API (registro)
col_usuarios.create_index('email', unique=True)


✅ Conexion a MongoDB Atlas exitosa
Documentos en precios_ohlcv: 1252
Documentos en predicciones: 0
Documentos en metricas_modelos: 0


## 4. App FastAPI con CORS habilitado

`allow_origins=['*']` es necesario porque el frontend vive en otro dominio (GitHub Pages) y el navegador bloquea por defecto las llamadas cross-origin.

In [31]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(title='Ernesto Investing AI - API')

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

def limpiar_documento(doc):
    """Elimina campos no serializables / internos antes de responder JSON."""
    if doc is None:
        return None
    doc.pop('_id', None)
    doc.pop('created_at', None)
    return doc

## 5. Endpoint 1 — `/api/salud`

In [32]:
@app.get('/api/salud')
def salud():
    """Verifica que el servidor y la base de datos esten funcionando."""
    try:
        cliente.admin.command('ping')
        db_ok = True
    except Exception:
        db_ok = False

    return {
        'estado': 'ok' if db_ok else 'error',
        'api': 'activa',
        'mongodb': 'conectado' if db_ok else 'desconectado',
    }

## 6. Endpoint 2 — `/api/mercado/{ticker}`
Devuelve el historico OHLCV + indicadores de un ticker (por defecto, los ultimos 100 registros, ordenados por fecha).

In [33]:
@app.get('/api/mercado/{ticker}')
def mercado(ticker: str, limite: int = 100):
    """Datos OHLCV con indicadores desde MongoDB para un ticker dado."""
    ticker = ticker.upper()

    cursor = (
        col_precios.find({'ticker': ticker})
        .sort('fecha', -1)
        .limit(limite)
    )
    documentos = [limpiar_documento(d) for d in cursor]

    if not documentos:
        raise HTTPException(status_code=404, detail=f'No hay datos de mercado para {ticker}')

    documentos.reverse()  # orden cronologico ascendente para graficar

    return {
        'ticker': ticker,
        'cantidad': len(documentos),
        'datos': documentos,
    }

## 7. Endpoint 3 — `/api/svc/{ticker}`
Devuelve la senal BUY/SELL mas reciente y las metricas del modelo SVC para un ticker.

In [34]:
@app.get('/api/svc/{ticker}')
def svc(ticker: str):
    """Senal actual y metricas del clasificador SVC desde MongoDB."""
    ticker = ticker.upper()

    prediccion = col_predicciones.find_one(
        {'ticker': ticker}, sort=[('fecha', -1)]
    )
    metricas = col_metricas.find_one(
        {'ticker': ticker}, sort=[('fecha', -1)]
    )

    if not prediccion and not metricas:
        raise HTTPException(status_code=404, detail=f'No hay predicciones para {ticker}')

    return {
        'ticker': ticker,
        'prediccion': limpiar_documento(prediccion),
        'metricas': limpiar_documento(metricas),
    }

## 8. Endpoint 4 — `/api/rnns/{ticker}`
Devuelve las 4 clasificaciones RNN (LSTM, BiLSTM, GRU, SimpleRNN) ya entrenadas por el Notebook 4.

In [ ]:
@app.get('/api/rnns/{ticker}')
def rnns(ticker: str):
    """Clasificaciones RNN (4 arquitecturas) desde MongoDB, pobladas por el Notebook 4."""
    ticker = ticker.upper()
    doc = col_rnns.find_one({'ticker': ticker})
    if not doc:
        raise HTTPException(
            status_code=404,
            detail=f'No hay clasificaciones RNN para {ticker}. Corre primero el Notebook 4.',
        )
    return limpiar_documento(doc)

## 9. Endpoint 5 — `/api/lstm/{ticker}`
Devuelve el pronóstico del regresor LSTM ya entrenado por el Notebook 5 (horizonte fijo de 30 días, quedó pre-calculado al guardarse en Mongo).

In [ ]:
@app.get('/api/lstm/{ticker}')
def lstm(ticker: str):
    """Pronóstico del regresor LSTM desde MongoDB, poblado por el Notebook 5."""
    ticker = ticker.upper()
    doc = col_lstm.find_one({'ticker': ticker})
    if not doc:
        raise HTTPException(
            status_code=404,
            detail=f'No hay pronóstico LSTM para {ticker}. Corre primero el Notebook 5.',
        )
    return limpiar_documento(doc)

## 10. Endpoints 6 y 7 — `/api/auth/registro` y `/api/auth/login`
Autenticación real contra MongoDB. Contraseñas con **PBKDF2-HMAC-SHA256** (200,000 iteraciones) + salt aleatorio por usuario — nunca se guarda la contraseña en texto plano.

In [ ]:
import hashlib
import os
from typing import Optional
from pydantic import BaseModel, EmailStr, Field


def hash_password(password: str, salt: Optional[str] = None):
    if salt is None:
        salt = os.urandom(16).hex()
    hash_hex = hashlib.pbkdf2_hmac('sha256', password.encode(), bytes.fromhex(salt), 200_000).hex()
    return hash_hex, salt


class RegistroRequest(BaseModel):
    nombre: str = Field(min_length=1)
    email: EmailStr
    password: str = Field(min_length=8)
    perfil: str


class LoginRequest(BaseModel):
    email: EmailStr
    password: str


@app.post('/api/auth/registro')
def auth_registro(payload: RegistroRequest):
    if payload.perfil not in ('conservador', 'moderado', 'agresivo'):
        raise HTTPException(status_code=400, detail='Perfil de riesgo invalido.')

    email = payload.email.lower()
    if col_usuarios.find_one({'email': email}):
        raise HTTPException(status_code=409, detail='Ya existe una cuenta con ese correo.')

    hash_hex, salt = hash_password(payload.password)
    col_usuarios.insert_one({
        'email': email,
        'nombre': payload.nombre,
        'perfil': payload.perfil,
        'password_hash': hash_hex,
        'salt': salt,
    })
    return {'mensaje': f'Cuenta creada para {payload.nombre}.'}


@app.post('/api/auth/login')
def auth_login(payload: LoginRequest):
    email = payload.email.lower()
    usuario = col_usuarios.find_one({'email': email})
    if not usuario:
        raise HTTPException(status_code=401, detail='Correo o contraseña incorrectos.')

    hash_calculado, _ = hash_password(payload.password, usuario['salt'])
    if hash_calculado != usuario['password_hash']:
        raise HTTPException(status_code=401, detail='Correo o contraseña incorrectos.')

    return {'nombre': usuario['nombre'], 'perfil': usuario['perfil'], 'email': email}

## 11. Endpoint 8 — `/api/portafolio/optimizar`
Única excepción a 'la API no recalcula nada': la optimización de Markowitz es barata (no es un entrenamiento de red neuronal), así que se calcula al vuelo con `scipy.optimize` sobre los retornos históricos guardados en `precios_ohlcv`.

In [ ]:
import math
import pandas as pd
from scipy.optimize import minimize

TICKERS = ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']
ACTIVOS_PORTAFOLIO = TICKERS + ['CASH']
TASA_LIBRE_RIESGO_ANUAL = 0.02


class ActivoPeso(BaseModel):
    ticker: str
    peso: float


class PortafolioRequest(BaseModel):
    activos: list[ActivoPeso]
    capital_total: float = 100000.0


def _retornos_diarios(ticker: str) -> pd.Series:
    cursor = col_precios.find({'ticker': ticker}).sort('fecha', 1)
    df = pd.DataFrame(list(cursor))
    if df.empty:
        return pd.Series(dtype=float)
    return df['Close'].pct_change().dropna()


def _stats_mercado():
    series = {tk: _retornos_diarios(tk) for tk in TICKERS}
    df_ret = pd.DataFrame(series).dropna(how='all').fillna(0.0)
    medias = df_ret.mean() * 252
    cov = df_ret.cov() * 252
    medias['CASH'] = TASA_LIBRE_RIESGO_ANUAL
    cov['CASH'] = 0.0
    cov.loc['CASH'] = 0.0
    orden = TICKERS + ['CASH']
    return medias[orden], cov.loc[orden, orden], df_ret


def _ratios_cartera(pesos, medias, cov, retornos_hist):
    retorno = float(np.dot(pesos, medias))
    riesgo = float(math.sqrt(max(np.dot(pesos, np.dot(cov, pesos)), 0.0)))
    sharpe = (retorno - TASA_LIBRE_RIESGO_ANUAL) / riesgo if riesgo > 1e-9 else 0.0
    serie = retornos_hist[TICKERS].fillna(0.0).dot(pesos[:len(TICKERS)])
    downside = serie[serie < 0]
    downside_std = float(downside.std() * math.sqrt(252)) if len(downside) > 1 else 0.0
    sortino = (retorno - TASA_LIBRE_RIESGO_ANUAL) / downside_std if downside_std > 1e-9 else 0.0
    return {'sharpe': round(sharpe, 3), 'sortino': round(sortino, 3)}, riesgo * 100, retorno * 100


def _optimizar_markowitz(medias, cov):
    n = len(medias)
    def neg_sharpe(w):
        r = np.dot(w, medias)
        v = math.sqrt(max(np.dot(w, np.dot(cov, w)), 1e-12))
        return -(r - TASA_LIBRE_RIESGO_ANUAL) / v
    restricciones = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
    limites = [(0.0, 0.6)] * n
    w0 = np.repeat(1.0 / n, n)
    res = minimize(neg_sharpe, w0, method='SLSQP', bounds=limites, constraints=restricciones)
    w = res.x if res.success else w0
    w = np.clip(w, 0, None)
    return w / w.sum()


@app.post('/api/portafolio/optimizar')
def portafolio_optimizar(payload: PortafolioRequest):
    medias, cov, retornos_hist = _stats_mercado()
    orden = TICKERS + ['CASH']
    medias_arr, cov_arr = medias.values, cov.values

    pesos_dict = {a.ticker: a.peso / 100.0 for a in payload.activos}
    pesos_actuales = np.array([pesos_dict.get(tk, 0.0) for tk in orden])
    if pesos_actuales.sum() <= 0:
        pesos_actuales = np.repeat(1.0 / len(orden), len(orden))
    else:
        pesos_actuales = pesos_actuales / pesos_actuales.sum()

    ratios_actual, riesgo_actual, retorno_actual = _ratios_cartera(
        pesos_actuales, medias_arr, cov_arr, retornos_hist)
    pesos_optimos = _optimizar_markowitz(medias_arr, cov_arr)
    ratios_optimo, riesgo_optimo, retorno_optimo = _ratios_cartera(
        pesos_optimos, medias_arr, cov_arr, retornos_hist)

    rng = np.random.default_rng(42)
    muestras = rng.dirichlet(np.ones(len(orden)) * 1.5, size=400)
    riesgos_nube = [math.sqrt(max(np.dot(m, np.dot(cov_arr, m)), 0.0)) * 100 for m in muestras]
    retornos_nube = [float(np.dot(m, medias_arr)) * 100 for m in muestras]

    return {
        'frontera': {'riesgo': riesgos_nube, 'retorno': retornos_nube},
        'actual': {'riesgo': round(riesgo_actual, 2), 'retorno': round(retorno_actual, 2)},
        'optimo': {
            'riesgo': round(riesgo_optimo, 2),
            'retorno': round(retorno_optimo, 2),
            'pesos': {tk: round(float(p) * 100, 2) for tk, p in zip(orden, pesos_optimos)},
        },
        'ratios': {'actual': ratios_actual, 'optimizado': ratios_optimo},
    }

## 12. Levantar el servidor y exponerlo con ngrok

Nota pyngrok 8.x: el authtoken se setea con `conf.get_default().auth_token`, no con `ngrok.set_auth_token()` (metodo viejo).

In [35]:
import nest_asyncio
import uvicorn
from pyngrok import ngrok, conf

nest_asyncio.apply()

conf.get_default().auth_token = NGROK_AUTHTOKEN

# Cerrar tuneles previos por si el notebook se re-ejecuta
ngrok.kill()

tunel_publico = ngrok.connect(8000)
print('🌐 URL publica de la API:', tunel_publico.public_url)
print('📄 Swagger UI disponible en:', tunel_publico.public_url + '/docs')
print('\nCopia la URL publica y pegala en el index.html del frontend.')

🌐 URL publica de la API: https://grape-likeness-cognitive.ngrok-free.dev
📄 Swagger UI disponible en: https://grape-likeness-cognitive.ngrok-free.dev/docs

Copia la URL publica y pegala en el index.html del frontend.


In [36]:
# Esta celda queda corriendo (bloquea) mientras la API esta activa.
# Para detenerla: Entorno de ejecucion > Interrumpir ejecucion.
import threading
import uvicorn
from pyngrok import ngrok, conf

# 1. Asegurarnos de que el token de ngrok esté configurado
conf.get_default().auth_token = NGROK_AUTHTOKEN

# 2. Limpiamos cualquier túnel previo
ngrok.kill()

# 3. Conectamos ngrok al puerto 8000
tunel_publico = ngrok.connect(8000)
print('🌐 URL pública de la API:', tunel_publico.public_url)
print('📄 Swagger UI disponible en:', tunel_publico.public_url + '/docs')
print('\nCopia la URL pública y pégala en el index.html del frontend.')

# 4. Magia: Envolvemos Uvicorn en una función
def run_server():
    # log_level="error" evita que tu notebook se llene de texto
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level="error")

# 5. Lo ejecutamos en un hilo separado
thread = threading.Thread(target=run_server)
thread.daemon = True
thread.start()

print("\n✅ ¡Servidor FastAPI ejecutándose perfectamente en segundo plano!")

🌐 URL pública de la API: https://grape-likeness-cognitive.ngrok-free.dev
📄 Swagger UI disponible en: https://grape-likeness-cognitive.ngrok-free.dev/docs

Copia la URL pública y pégala en el index.html del frontend.

✅ ¡Servidor FastAPI ejecutándose perfectamente en segundo plano!


ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use


## 13. (Opcional) Prueba rapida desde el propio notebook

Si querés probar la API sin salir de Colab, abrí una **celda nueva en otro notebook** (o corré esta antes de bloquear con `uvicorn.run`) usando `requests` contra la URL de `tunel_publico.public_url`. Ejemplo:

```python
import requests
base = tunel_publico.public_url
print(requests.get(f'{base}/api/salud').json())
print(requests.get(f'{base}/api/mercado/BVN').json())
print(requests.get(f'{base}/api/svc/BVN').json())
```

In [37]:
import requests
base = tunel_publico.public_url
print(requests.get(f'{base}/api/salud').json())
print(requests.get(f'{base}/api/mercado/BVN').json())
print(requests.get(f'{base}/api/svc/BVN').json())

{'estado': 'ok', 'api': 'activa', 'mongodb': 'conectado'}
{'ticker': 'BVN', 'cantidad': 100, 'datos': [{'ticker': 'BVN', 'fecha': '2026-02-09', 'Close': 38.015987396240234, 'High': 38.288087993756356, 'Low': 36.587475014209545, 'Open': 36.587475014209545, 'Volume': 1386000.0, 'SMA_20': 35.369832801818845, 'SMA_50': 30.36517230987549, 'EMA_12': 35.53054796053785, 'EMA_26': 33.88532769662814, 'RSI_14': 54.08871747240364}, {'ticker': 'BVN', 'fecha': '2026-02-10', 'Close': 37.909088134765625, 'High': 38.21034088758875, 'Low': 37.27743415935442, 'Open': 37.6564272860095, 'Volume': 906200.0, 'SMA_20': 35.59334182739258, 'SMA_50': 30.651458282470703, 'EMA_12': 35.89647721811136, 'EMA_26': 34.18338402537906, 'RSI_14': 55.421684674921636}, {'ticker': 'BVN', 'fecha': '2026-02-11', 'Close': 39.33760452270508, 'High': 39.561113196993865, 'Low': 37.81191213648548, 'Open': 39.38619272023884, 'Volume': 1357600.0, 'SMA_20': 35.89167785644531, 'SMA_50': 30.95659679412842, 'EMA_12': 36.425881418818086, 